In [1]:
!pip install -q plotly pandas

import plotly
print(f"Plotly version: {plotly.__version__}")


Plotly version: 5.24.1


In [2]:
import pandas as pd
import numpy as np

# Upload your data file
from google.colab import files
print("Upload your drug-gene data file (tab-separated with columns: drug, gene, risk_association, paired_tstat, p)")
uploaded = files.upload()
filename = list(uploaded.keys())[0]

# Read data
df = pd.read_csv(filename, sep='\t')
print(f"\nLoaded {len(df)} rows from {filename}")
print(f"Columns: {list(df.columns)}")
print(f"Drugs: {df['drug'].nunique()} unique")
print(f"Genes: {df['gene'].nunique()} unique")

# Filter: p < 0.01 and concordant direction
# High risk = negative t-stat, Low risk = positive t-stat
filtered = df[
    (df['p'] < 0.01) &
    (
        ((df['paired_tstat'] < 0) & (df['risk_association'] == 'High_risk')) |
        ((df['paired_tstat'] > 0) & (df['risk_association'] == 'Low_risk'))
    )
].copy()

print(f"\nAfter filtering (p < 0.01, concordant direction): {len(filtered)} associations")
print(f"  High risk: {len(filtered[filtered['risk_association'] == 'High_risk'])}")
print(f"  Low risk: {len(filtered[filtered['risk_association'] == 'Low_risk'])}")
print(f"  Drugs: {filtered['drug'].nunique()}")
print(f"  Genes: {filtered['gene'].nunique()}")

Upload your drug-gene data file (tab-separated with columns: drug, gene, risk_association, paired_tstat, p)


Saving example.txt to example.txt

Loaded 2418 rows from example.txt
Columns: ['drug', 'gene', 'risk_association', 'paired_tstat', 'p']
Drugs: 13 unique
Genes: 186 unique

After filtering (p < 0.01, concordant direction): 177 associations
  High risk: 73
  Low risk: 104
  Drugs: 8
  Genes: 114


In [3]:
import plotly.graph_objects as go

# Define drug colors — muted, professional palette
drug_colors = {
    'AM095':        {'main': '#c0392b', 'link': 'rgba(192,57,43,0.25)'},
    'Cenicriviroc': {'main': '#2980b9', 'link': 'rgba(41,128,185,0.25)'},
    'EGCG':         {'main': '#27ae60', 'link': 'rgba(39,174,96,0.25)'},
    'Erlotinib':    {'main': '#7f8c8d', 'link': 'rgba(127,140,141,0.22)'},
    'Galunisertib': {'main': '#16a085', 'link': 'rgba(22,160,133,0.25)'},
    'Metformin':    {'main': '#8e44ad', 'link': 'rgba(142,68,173,0.25)'},
    'MG132':        {'main': '#d35400', 'link': 'rgba(211,84,0,0.25)'},
    'Pioglitazone': {'main': '#2c3e50', 'link': 'rgba(44,62,80,0.25)'},
}

# Build node lists: high-risk genes (left) | drugs (center) | low-risk genes (right)
high_genes = sorted(filtered[filtered['risk_association'] == 'High_risk']['gene'].unique())
drugs = sorted(filtered['drug'].unique())
low_genes = sorted(filtered[filtered['risk_association'] == 'Low_risk']['gene'].unique())

nodes = list(high_genes) + list(drugs) + list(low_genes)
node_index = {name: i for i, name in enumerate(nodes)}

# Node colors
node_colors = []
for n in nodes:
    if n in drug_colors:
        node_colors.append(drug_colors[n]['main'])
    elif n in high_genes:
        node_colors.append('#c0392b')
    else:
        node_colors.append('#27ae60')

# Build links
sources = []
targets = []
values = []
link_colors = []
link_labels = []

for _, row in filtered.iterrows():
    drug = row['drug']
    gene = row['gene']
    risk = row['risk_association']
    tstat = row['paired_tstat']
    pval = row['p']
    abs_t = abs(tstat)

    color = drug_colors.get(drug, {'main': '#888', 'link': 'rgba(136,136,136,0.2)'})

    if risk == 'High_risk':
        sources.append(node_index[gene])
        targets.append(node_index[drug])
        values.append(abs_t)
        link_colors.append(color['link'])
        link_labels.append(f"{gene} → {drug} | High risk | t={tstat:.3f} | p={pval:.2e}")
    else:
        sources.append(node_index[drug])
        targets.append(node_index[gene])
        values.append(abs_t)
        link_colors.append(color['link'])
        link_labels.append(f"{drug} → {gene} | Low risk | t={tstat:.3f} | p={pval:.2e}")

# Create Sankey figure
fig = go.Figure(data=[go.Sankey(
    arrangement='snap',
    node=dict(
        pad=6,
        thickness=14,
        line=dict(color='rgba(0,0,0,0.1)', width=0.5),
        label=nodes,
        color=node_colors,
        hovertemplate='<b>%{label}</b><br>Connections: %{value:.1f}<extra></extra>'
    ),
    link=dict(
        source=sources,
        target=targets,
        value=values,
        color=link_colors,
        customdata=link_labels,
        hovertemplate='%{customdata}<extra></extra>'
    )
)])

plot_height = max(500, min(1200, len(nodes) * 12))

fig.update_layout(
    font=dict(family='Source Sans 3, Helvetica, Arial, sans-serif', size=10, color='#444'),
    paper_bgcolor='#fff',
    plot_bgcolor='#fff',
    margin=dict(l=5, r=5, t=5, b=5),
    height=plot_height,
    width=900
)

fig.show()
print(f"\n✓ Sankey plot rendered with {len(nodes)} nodes and {len(sources)} links")




✓ Sankey plot rendered with 122 nodes and 177 links


In [4]:
# Build the full HTML with controls, stats, and styling
html_template = """<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="utf-8">
<title>Drug-Gene Risk Association — Sankey Plot</title>
<script src="https://cdn.plot.ly/plotly-2.32.0.min.js"></script>
<link href="https://fonts.googleapis.com/css2?family=Source+Sans+3:wght@400;600;700&family=IBM+Plex+Mono:wght@400;500&display=swap" rel="stylesheet">
<style>
  * { margin: 0; padding: 0; box-sizing: border-box; }
  body { font-family: 'Source Sans 3', sans-serif; background: #f7f8fa; color: #333; min-height: 100vh; }
  .container { max-width: 960px; margin: 0 auto; padding: 20px; }
  .header { text-align: center; padding: 25px 20px 8px; }
  .header h1 { font-size: 22px; font-weight: 700; color: #2c3e50; }
  .header .subtitle { font-size: 13px; color: #7f8c8d; margin-top: 4px; font-family: 'IBM Plex Mono', monospace; }
  .controls { display: flex; justify-content: center; gap: 8px; padding: 12px 20px; flex-wrap: wrap; }
  .control-btn { padding: 6px 14px; border: 1px solid #ddd; border-radius: 6px; background: #fff; color: #555; font-family: 'Source Sans 3', sans-serif; font-size: 12px; cursor: pointer; transition: all 0.2s; }
  .control-btn:hover { background: #f0f0f0; border-color: #bbb; }
  .control-btn.active { background: #2c3e50; border-color: #2c3e50; color: #fff; }
  #sankey-plot { width: 100%; background: #fff; border: 1px solid #e8e8e8; border-radius: 8px; box-shadow: 0 1px 4px rgba(0,0,0,0.06); }
  .stats-bar { display: flex; justify-content: center; gap: 20px; padding: 10px 20px; flex-wrap: wrap; }
  .stat-card { text-align: center; min-width: 80px; }
  .stat-card .number { font-size: 20px; font-weight: 700; font-family: 'IBM Plex Mono', monospace; }
  .stat-card .label { font-size: 10px; color: #999; text-transform: uppercase; letter-spacing: 0.8px; }
  .legend { display: flex; justify-content: center; gap: 24px; padding: 8px 20px; }
  .legend-item { display: flex; align-items: center; gap: 6px; font-size: 12px; color: #666; }
  .legend-dot { width: 10px; height: 10px; border-radius: 2px; }
  .footer { text-align: center; padding: 8px; font-size: 11px; color: #aaa; font-family: 'IBM Plex Mono', monospace; }
</style>
</head>
<body>
<div class="container">
<div class="header">
  <h1>Drug–Gene Risk Association Network</h1>
  <div class="subtitle">Paired t-test | FDR &lt; 0.01 | Hepatocellular Carcinoma Pharmacogenomics</div>
</div>
<div class="controls">
  <button class="control-btn active" onclick="showAll()">All Drugs</button>
  DRUG_BUTTONS_PLACEHOLDER
</div>
<div class="legend">
  <div class="legend-item"><div class="legend-dot" style="background:#c0392b"></div> High-risk genes (left)</div>
  <div class="legend-item"><div class="legend-dot" style="background:#555"></div> Drugs (center)</div>
  <div class="legend-item"><div class="legend-dot" style="background:#27ae60"></div> Low-risk genes (right)</div>
</div>
<div id="stats-bar" class="stats-bar"></div>
<div id="sankey-plot"></div>
<div class="footer">Hover over links for t-statistic and p-value • Click drug buttons to filter • Drag nodes to rearrange</div>
</div>
<script>
const allData = DATA_PLACEHOLDER;
const drugColors = COLORS_PLACEHOLDER;

function buildSankey(filterDrug) {
  const filtered = filterDrug ? allData.filter(d => d.drug === filterDrug) : allData;
  const drugs = [...new Set(filtered.map(d => d.drug))];
  const highGenes = [...new Set(filtered.filter(d => d.risk === 'High_risk').map(d => d.gene))].sort();
  const lowGenes = [...new Set(filtered.filter(d => d.risk === 'Low_risk').map(d => d.gene))].sort();
  const nodes = [...highGenes, ...drugs, ...lowGenes];
  const ni = {}; nodes.forEach((n, i) => ni[n] = i);
  const src = [], tgt = [], val = [], lc = [], ll = [];
  filtered.forEach(d => {
    const at = Math.abs(d.tstat);
    const c = drugColors[d.drug] || {main:'#888',link:'rgba(136,136,136,0.2)'};
    if (d.risk === 'High_risk') { src.push(ni[d.gene]); tgt.push(ni[d.drug]); }
    else { src.push(ni[d.drug]); tgt.push(ni[d.gene]); }
    val.push(at); lc.push(c.link);
    ll.push(`<b>${d.gene}</b> – <b>${d.drug}</b><br>${d.risk.replace('_',' ')}<br>t = ${d.tstat.toFixed(3)}<br>p = ${d.p.toExponential(2)}`);
  });
  const nc = nodes.map(n => { if(drugColors[n]) return drugColors[n].main; if(highGenes.includes(n)) return '#c0392b'; return '#27ae60'; });
  Plotly.react('sankey-plot', [{type:'sankey',arrangement:'snap',node:{pad:6,thickness:14,line:{color:'rgba(0,0,0,0.1)',width:0.5},label:nodes,color:nc,hovertemplate:'<b>%{label}</b><br>Connections: %{value:.1f}<extra></extra>'},link:{source:src,target:tgt,value:val,color:lc,customdata:ll,hovertemplate:'%{customdata}<extra></extra>'}}],{font:{family:'Source Sans 3,sans-serif',size:10,color:'#444'},paper_bgcolor:'#fff',plot_bgcolor:'#fff',margin:{l:5,r:5,t:5,b:5},height:Math.max(500,Math.min(1200,nodes.length*12)),width:900},{displayModeBar:false,responsive:true});
  const hc=filtered.filter(d=>d.risk==='High_risk').length, lo=filtered.filter(d=>d.risk==='Low_risk').length;
  document.getElementById('stats-bar').innerHTML=`<div class="stat-card"><div class="number" style="color:#2c3e50">${new Set(filtered.map(d=>d.drug)).size}</div><div class="label">Drugs</div></div><div class="stat-card"><div class="number" style="color:#555">${new Set(filtered.map(d=>d.gene)).size}</div><div class="label">Genes</div></div><div class="stat-card"><div class="number" style="color:#c0392b">${hc}</div><div class="label">High Risk</div></div><div class="stat-card"><div class="number" style="color:#27ae60">${lo}</div><div class="label">Low Risk</div></div><div class="stat-card"><div class="number" style="color:#7f8c8d">${filtered.length}</div><div class="label">Total</div></div>`;
}
function showAll(){document.querySelectorAll('.control-btn').forEach(b=>b.classList.remove('active'));document.querySelector('.control-btn').classList.add('active');buildSankey(null);}
function filterDrug(d){document.querySelectorAll('.control-btn').forEach(b=>{b.classList.toggle('active',b.textContent===d)});buildSankey(d);}
buildSankey(null);
</script>
</body>
</html>"""

import json

# Prepare data for embedding in HTML
json_data = json.dumps([{
    'drug': r['drug'], 'gene': r['gene'], 'risk': r['risk_association'],
    'tstat': round(r['paired_tstat'], 6), 'p': round(r['p'], 8)
} for _, r in filtered.iterrows()])

json_colors = json.dumps({k: v for k, v in drug_colors.items()})

# Generate drug buttons
drug_buttons = '\n  '.join([
    f'<button class="control-btn" onclick="filterDrug(\'{d}\')">{d}</button>'
    for d in sorted(filtered['drug'].unique())
])

# Fill template
html_output = html_template.replace('DATA_PLACEHOLDER', json_data)
html_output = html_output.replace('COLORS_PLACEHOLDER', json_colors)
html_output = html_output.replace('DRUG_BUTTONS_PLACEHOLDER', drug_buttons)

# Save HTML
with open('sankey_drug_gene_network.html', 'w') as f:
    f.write(html_output)

print("✓ Saved: sankey_drug_gene_network.html")
print(f"  {len(filtered)} associations | {filtered['drug'].nunique()} drugs | {filtered['gene'].nunique()} genes")



✓ Saved: sankey_drug_gene_network.html
  177 associations | 8 drugs | 114 genes


In [5]:
from google.colab import files
files.download('sankey_drug_gene_network.html')
print("✓ Download complete")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✓ Download complete
